# CoraML Facial-Walk Experiment

Train the transformer on facial walks, generate synthetic walks, reconstruct a graph, and compare link-prediction and graph statistics against the true CoraML graph.

## Ready To Run On T4

- This notebook is ready for a `Colab T4 GPU` runtime, not a TPU runtime.
- The training code uses `PyTorch + Hugging Face GPT-2`, so the intended accelerator is `cuda`.
- The graph split is `10% validation + 5% test`, while keeping the training graph connected.
- This full-run config targets about `1.0M` training chunks using `num_sign_configs=500`, `vertex_context_size=16`, and non-overlapping `dart_stride=7`.
- Validation uses `100k` generated walks per epoch; final evaluation uses `500k`.
- Checkpoints are written every epoch to `checkpoints/coraml_t4_run`, and the final model is written to `checkpoints/coraml_t4_run/final`.
- If the runtime disconnects, rerunning the training cell with `resume_from_latest=True` resumes from the newest saved epoch.


## Runtime Bootstrap

On Colab/remote kernels, run this first. It can clone the repo onto the remote runtime, switch into it, and mount Drive for persistent checkpoints. Locally it just keeps the current repo root.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

repo_url = 'https://github.com/rbmuk/facialgen.git'
repo_dir_name = 'facialgen'

running_in_colab = False
drive_root = None

try:
    from google.colab import drive  # type: ignore
    running_in_colab = True
except ImportError:
    drive = None

if running_in_colab:
    runtime_repo_root = Path('/content') / repo_dir_name
    if not runtime_repo_root.exists():
        subprocess.run(['git', 'clone', repo_url, str(runtime_repo_root)], check=True)
    os.chdir(runtime_repo_root)
    print('cwd =', Path.cwd())
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive')
    default_save_dir = drive_root / 'facialgen_checkpoints' / 'coraml_t4_run'
else:
    runtime_repo_root = Path.cwd().resolve()
    default_save_dir = runtime_repo_root / 'checkpoints' / 'coraml_t4_run'
    print('cwd =', runtime_repo_root)

default_save_dir = str(default_save_dir)
print('default_save_dir =', default_save_dir)

requirements_path = runtime_repo_root / 'requirements.txt'
if requirements_path.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
    print('Installed dependencies from', requirements_path)
else:
    print('No requirements.txt found at', requirements_path)


cwd = /Users/rbmuk/Documents/GDL/facialgen
default_save_dir = /Users/rbmuk/Documents/GDL/facialgen/checkpoints/coraml_t4_run
Installed dependencies from /Users/rbmuk/Documents/GDL/facialgen/requirements.txt


## Imports And Config

Import the package, reload local modules, and define the live experiment configuration.


In [ ]:
import importlib
from types import SimpleNamespace

import numpy as np
import pandas as pd
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'facialgen').is_dir():
            return candidate
    raise RuntimeError(
        'Could not locate repo root containing pyproject.toml and the facialgen package. '
        'Run the Runtime Bootstrap cell first, and on Colab set repo_url to your GitHub repository.'
    )

repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('repo_root =', repo_root)

import facialgen
import facialgen.data as data_mod
import facialgen.early_stopping as early_stopping
import facialgen.evaluation as evaluation
import facialgen.models as models
import facialgen.train as train_mod

importlib.invalidate_caches()
importlib.reload(data_mod)
importlib.reload(early_stopping)
importlib.reload(evaluation)
importlib.reload(models)
importlib.reload(train_mod)
importlib.reload(facialgen)

from facialgen.data import CyclicFaceChunkDataset
from facialgen.early_stopping import (
    connected_link_prediction_split,
    edge_overlap_ratio,
    link_prediction_scores_from_walks,
    sample_model_walks,
)
from facialgen.evaluation import (
    compute_graph_statistics,
    reconstruct_graph_from_generated_walks,
)
from facialgen.models import FacialGen
from facialgen.train import build_training_objects, resolve_device, seed_everything, train_model

print('CyclicFaceChunkDataset from:', CyclicFaceChunkDataset.__module__)

args = SimpleNamespace(
    dataset_name='coraml',
    seed=2026,
    data_dir='data',
    num_sign_configs=500,
    sign_seed=2026,
    epoch_seed=99,
    vertex_context_size=16,
    dart_stride=7,
    batch_size=64,
    epochs=20,
    lr=3e-4,
    weight_decay=0.01,
    grad_clip=1.0,
    num_workers=0,
    device='auto',
    n_layer=1,
    n_head=4,
    n_embd=256,
    dropout=0.0,
    save_dir=default_save_dir,
    resume_from_latest=True,
    log_every=20,
    early_stop_mode='val',
    early_stop_patience=5,
    early_stop_min_delta=0.0,
    val_fraction=0.10,
    test_fraction=0.05,
    split_seed=123,
    eval_generated_walks=100_000,
    eval_max_length=None,
    target_edge_overlap=0.5,
    use_link_prediction_split=True,
)

checkpoint_dir = None
num_generated_graphs = 1
final_generated_walks = 500_000
final_max_length = args.vertex_context_size
generation_batch_size = 256
reconstruction_seed = 777

seed_everything(args.seed)
print(f"seed = {args.seed}")

device = resolve_device(args.device)
print(f"dataset = {args.dataset_name}")
print(f"vertex_context_size = {args.vertex_context_size}, dart_stride = {args.dart_stride}")
print(f"FAST baseline config for CoraML: n_layer = {args.n_layer}")
print(f"checkpoint_dir = {args.save_dir}")
approx_chunks_per_sign_config = 1_980
approx_total_chunks = approx_chunks_per_sign_config * args.num_sign_configs
print(f"approx chunk budget ~= {approx_total_chunks:,} (~{approx_chunks_per_sign_config} per sign config)")
print(f"resume_from_latest = {args.resume_from_latest}")
print('device =', device)


repo_root = /Users/rbmuk/Documents/GDL/facialgen
CyclicFaceChunkDataset from: facialgen.data
seed = 2026
dataset = coraml
context_size = 16, stride = 7
FAST baseline config for CoraML: n_layer = 1
checkpoint_dir = /Users/rbmuk/Documents/GDL/facialgen/checkpoints/coraml_t4_run
resume_from_latest = True
device = mps


## Reproducibility

Rerunning the import/config cell above now reloads `facialgen` and reseeds Python, NumPy, and PyTorch through `seed_everything(args.seed)`.

For non-overlapping dart-window sampling with the faithful facial-walk encoding, use `dart_stride = (vertex_context_size - 1) // 2`.


In [ ]:
approx_chunks_per_sign_config = 1980
approx_total_chunks = args.num_sign_configs * approx_chunks_per_sign_config

print(f"seed = {args.seed}")
print(f"dataset = {args.dataset_name}")
print(f"vertex_context_size = {args.vertex_context_size}, dart_stride = {args.dart_stride}")
print(f"num_sign_configs = {args.num_sign_configs}")
print(f"eval_generated_walks = {args.eval_generated_walks:,}")
print(f"final_generated_walks = {final_generated_walks:,}")
print(f"save_dir = {args.save_dir}")
print(f"resume_from_latest = {args.resume_from_latest}")
print(f"approx total chunk samples ~= {approx_total_chunks:,}")


## Dataset Preview

Build the training objects once and inspect dataset size, dart stride, and chunk counts before training.


In [ ]:
chunk_ds_preview, loader_preview, model_preview, eval_info_preview = build_training_objects(args)
print(f"live dart_stride = {chunk_ds_preview.dart_stride}")
print(f"num full face sequences = {len(chunk_ds_preview.face_dataset)}")
print(f"num chunk samples = {len(chunk_ds_preview)}")


## Data Visualization

This section visualizes how one facial walk is rotated in dart space and then converted into BOS-anchored training chunks.


In [ ]:
demo_epoch = 0
chunk_ds_preview.set_epoch(demo_epoch)
demo_face_indices = [1, len(chunk_ds_preview.face_dataset) - 1]

for demo_face_idx in demo_face_indices:
    dart_face = chunk_ds_preview.face_dataset.dart_faces[demo_face_idx]
    rotated_dart_face = chunk_ds_preview._rotated_dart_face(demo_face_idx)
    full_faithful_vertex_face = chunk_ds_preview.face_dataset.sequences[demo_face_idx]

    matching_indices = [
        idx
        for idx, (face_idx, _, _) in enumerate(chunk_ds_preview.chunk_to_face)
        if face_idx == demo_face_idx
    ]
    demo_indices = matching_indices[:5]
    demo_rows = []
    for idx in demo_indices:
        item = chunk_ds_preview[idx]
        demo_rows.append({
            'chunk_index': int(item['chunk_index']),
            'chunk_start_dart': int(item['chunk_start']),
            'dart_length': int(item['dart_length']),
            'has_eos': bool(item['has_eos']),
            'tokens': item['tokens'].tolist(),
            'vertex_tokens_wo_bos': item['tokens'].tolist()[1:],
        })

    print('face_index =', demo_face_idx)
    print('epoch =', demo_epoch)
    print('full dart-face length =', len(dart_face))
    print('rotated dart-face length =', len(rotated_dart_face))
    print('full faithful vertex length =', len(full_faithful_vertex_face))
    print('BOS token id =', chunk_ds_preview.face_dataset.bos_token_id)
    print('EOS token id =', chunk_ds_preview.face_dataset.eos_token_id)
    print()
    print('first 12 darts of rotated face:')
    print([(int(u), int(v)) for (u, v) in rotated_dart_face[:12]])
    print()
    display(pd.DataFrame(demo_rows))
    print('-' * 80)

all_dart_lengths = np.array([
    int(chunk_ds_preview[idx]['dart_length'])
    for idx in range(len(chunk_ds_preview))
], dtype=int)
all_has_eos = np.array([
    bool(chunk_ds_preview[idx]['has_eos'])
    for idx in range(len(chunk_ds_preview))
], dtype=bool)

plt.figure(figsize=(8, 4))
plt.hist(
    all_dart_lengths,
    bins=np.arange(1, all_dart_lengths.max() + 2) - 0.5,
    color='steelblue',
    alpha=0.85,
)
plt.title('Dart-length distribution across all training chunks in one epoch')
plt.xlabel('darts per chunk')
plt.ylabel('count')
plt.xticks(range(1, all_dart_lengths.max() + 1))
plt.grid(alpha=0.2)
plt.show()

summary = pd.DataFrame([
    {
        'min_dart_length': int(all_dart_lengths.min()),
        'median_dart_length': float(np.median(all_dart_lengths)),
        'mean_dart_length': float(all_dart_lengths.mean()),
        'max_dart_length': int(all_dart_lengths.max()),
        'num_chunks': int(all_dart_lengths.size),
        'num_chunks_with_EOS': int(all_has_eos.sum()),
    }
])
display(summary)


## Training

Train from scratch or resume from the latest checkpoint, then keep the trained model in memory for downstream evaluation.


In [ ]:
if checkpoint_dir is None:
    model, eval_info, history = train_model(args)
else:
    _, _, _, eval_info = build_training_objects(args)
    model = FacialGen.from_pretrained(checkpoint_dir)
    history = []

model.to(device)
print(type(model).__name__)


## Training Curves

Visualize NLL/perplexity and validation metrics recorded in the training history.


In [ ]:
history_df = pd.DataFrame(history)
display(history_df)
if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['mean_nll'], marker='o', label='train NLL')
    axes[0].set_title('Training NLL by Epoch')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('mean_nll')
    axes[0].grid(True, alpha=0.3)
    if 'val_roc_auc' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_roc_auc'], marker='o', label='val ROC-AUC')
    if 'val_ap' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_ap'], marker='o', label='val AP')
    if 'val_score' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_score'], marker='o', label='val score')
    axes[1].set_title('Validation Metrics by Epoch')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('score')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

print('Final model checkpoint:', Path(args.save_dir) / 'final' if args.save_dir else 'not saved')


## Sample Inspection

Load a saved checkpoint and inspect a few generated sequences directly.


In [ ]:
checkpoint_root = Path(args.save_dir) if args.save_dir else None
latest_epoch_dir = None
if checkpoint_root is not None and checkpoint_root.exists():
    epoch_dirs = sorted(checkpoint_root.glob('epoch_*'))
    if epoch_dirs:
        latest_epoch_dir = epoch_dirs[-1]

inspect_checkpoint_dir = (
    Path(checkpoint_dir)
    if checkpoint_dir is not None
    else (Path(args.save_dir) / 'final' if args.save_dir and (Path(args.save_dir) / 'final').exists() else latest_epoch_dir)
)
inspect_num_samples = 5
inspect_max_length = args.vertex_context_size

if inspect_checkpoint_dir is None:
    raise RuntimeError('No checkpoint directory is available to inspect.')

print('inspect_checkpoint_dir =', inspect_checkpoint_dir)
inspect_model = FacialGen.from_pretrained(str(inspect_checkpoint_dir))
inspect_model.to(device)
inspect_walks = sample_model_walks(
    inspect_model,
    num_samples=inspect_num_samples,
    max_length=inspect_max_length,
    bos_token_id=int(eval_info['bos_token_id']),
    eos_token_id=int(eval_info['eos_token_id']) if eval_info['eos_token_id'] is not None else None,
    device=device,
    batch_size=inspect_num_samples,
    show_progress=False,
)

rows = []
for idx, seq in enumerate(inspect_walks):
    special_tokens = {int(eval_info['bos_token_id'])}
    if eval_info['eos_token_id'] is not None:
        special_tokens.add(int(eval_info['eos_token_id']))
    stripped = [tok for tok in seq if tok not in special_tokens]
    rows.append({
        'sample_id': idx,
        'raw_tokens': seq,
        'vertex_tokens': stripped,
        'length_with_specials': len(seq),
        'vertex_length': len(stripped),
    })

display(pd.DataFrame(rows))


## Final Evaluation

Generate walks from the trained model, reconstruct a graph, and compare link prediction and graph statistics against the true CoraML LCC.


In [ ]:
reference_adj = eval_info['reference_adj']
reference_labels = eval_info['reference_labels']
num_nodes = int(eval_info['num_nodes'])
num_reference_edges = int(eval_info['num_reference_edges'])
lp_split = eval_info['link_prediction_split']
if lp_split is None:
    lp_split = connected_link_prediction_split(
        reference_adj,
        val_fraction=args.val_fraction,
        test_fraction=args.test_fraction,
        seed=args.split_seed,
    )

reference_stats = compute_graph_statistics(reference_adj, labels=reference_labels)

generated_results = []
generated_stats_rows = []

for graph_idx in range(num_generated_graphs):
    walks = sample_model_walks(
        model,
        num_samples=final_generated_walks,
        max_length=final_max_length,
        bos_token_id=int(eval_info['bos_token_id']),
        eos_token_id=int(eval_info['eos_token_id']) if eval_info['eos_token_id'] is not None else None,
        device=device,
        batch_size=generation_batch_size,
        show_progress=True,
        progress_desc=f'final sampling graph {graph_idx + 1}/{num_generated_graphs}',
    )

    A_hat, S = reconstruct_graph_from_generated_walks(
        walks,
        num_nodes=num_nodes,
        target_num_edges=num_reference_edges,
        seed=reconstruction_seed + graph_idx,
    )

    val_scores = link_prediction_scores_from_walks(
        walks,
        num_nodes=num_nodes,
        positive_edges=lp_split['val_edges'],
        negative_edges=lp_split['val_non_edges'],
    )
    test_scores = link_prediction_scores_from_walks(
        walks,
        num_nodes=num_nodes,
        positive_edges=lp_split['test_edges'],
        negative_edges=lp_split['test_non_edges'],
    )
    graph_stats = compute_graph_statistics(A_hat, labels=reference_labels)
    overlap = edge_overlap_ratio(A_hat, reference_adj)

    generated_results.append({
        'graph_id': graph_idx,
        'val_roc_auc': float(val_scores['roc_auc']),
        'val_ap': float(val_scores['average_precision']),
        'test_roc_auc': float(test_scores['roc_auc']),
        'test_ap': float(test_scores['average_precision']),
        'edge_overlap': float(overlap),
    })
    generated_stats_rows.append(graph_stats)

lp_table = pd.DataFrame(generated_results)
display(lp_table)

metric_names = list(reference_stats.keys())
stats_table = pd.DataFrame([
    {
        'metric': metric,
        'true_coraml': float(reference_stats[metric]),
        'generated_mean': float(np.nanmean([row[metric] for row in generated_stats_rows])),
        'abs_diff': abs(
            float(np.nanmean([row[metric] for row in generated_stats_rows]))
            - float(reference_stats[metric])
        ),
    }
    for metric in metric_names
])
display(stats_table)

if history:
    display(pd.DataFrame(history))
